In [9]:
from glob import glob
from pathlib import Path
import pickle

import matplotlib.pyplot as plt
import numpy as np

from model_tuner.opt.regimes import NetRegime1D, NetRegime1DList
from model_tuner.opt.uc_mappers import NetUCMapper1D

from model_tuner.main import (
    UCMapFitParams,
    init_uc_mapper,
    plot_opt_iteration_pop

)

from model_tuner.utils import load_yaml

In [10]:
dirpath_base = Path(
    r'D:\WORK\Salvador\repo\model_tuner\test_data\main'
    r'\test_opt_A1_hpc_batch_qsub\experiments'
    #r'\test_2_pfr=(0.4_1.0_4)_wmult=0.005_alpha=0.2'
    r'\test_2_pfr=(0.4_1.0_4)_wmult=0.005_alpha=1'
)
dirpath_info = dirpath_base / 'info'

#pop_name_vis = 'SOM6'

step_num = 0

# Load Ru and Rc data
iter_data = []
n_iter = len(glob(str(dirpath_info / 'Ru_Rc_req_*_*.pkl')))
for n in range(n_iter):
    fpath_mask = str(dirpath_info / f'Ru_Rc_req_{n}_*.pkl')
    fpath_iter_data = glob(fpath_mask)[0]
    with open(fpath_iter_data, 'rb') as f:
        iter_data.append(pickle.load(f))

if step_num < 0:
    step_num = n_iter + step_num

# Extract data
if step_num > 0:
    Ru_prev_mat = iter_data[step_num - 1]['Ru']
    Rc_prev_mat = iter_data[step_num - 1]['Rc']
else:
    Ru_prev_mat = iter_data[step_num]['Ru']
    Rc_prev_mat = iter_data[step_num]['Ru']
Ru_mat = iter_data[step_num]['Ru']
Rc_mat = iter_data[step_num]['Rc']

pop_names = iter_data[-1]['pop_names']

Ru_prev_lst = NetRegime1DList.from_regimes_mat(pop_names, Ru_prev_mat)
Rc_prev_lst = NetRegime1DList.from_regimes_mat(pop_names, Rc_prev_mat)
Ru_lst = NetRegime1DList.from_regimes_mat(pop_names, Ru_mat)
Rc_lst = NetRegime1DList.from_regimes_mat(pop_names, Rc_mat)

#rc_min = Rc_lst[0][pop_name_vis].value
#rc_max = Rc_lst[-1][pop_name_vis].value


In [11]:
# Load uc map params
uc_map_params: UCMapFitParams = load_yaml(
    dirpath_base / 'uc_map_params.yaml',
    data_class=UCMapFitParams
)

uc_map_params.map_type = 'richards_1d'
uc_map_params.fit_param_bounds = {
    #'c': (0, rc_min),
    'a': (0, np.inf),
    'q': (1, 30),
    #'b': (0, np.inf)
}

uc_mapper = init_uc_mapper(uc_map_params)

alpha_mult = 0.9
alpha = 1

while alpha > 0.01:
    # Mix previous and new regimes
    Rc_lst_mixed = NetRegime1DList.mix(Rc_prev_lst, Rc_lst, alpha)

    try:
        # Fit uc mapper
        uc_mapper.set_to_identity()
        res = uc_mapper.fit_from_data(
            Ru_lst, Rc_lst_mixed,
            fit_params=uc_map_params.map_fit_params,
            bounds=uc_map_params.fit_param_bounds,
            verbose=0
        )
        if not res:
            print(f'FItting returned 0 with alpha={alpha:.04f}')
            alpha *= alpha_mult
            continue
    except Exception as e:
        print(f'Fitting failed with alpha={alpha:.04f}')
        break
        alpha *= alpha_mult
        continue

    # Apply Rc->Ru mapping
    Ru_lst_hat = uc_mapper.Rc_to_Ru(Rc_prev_lst)

    # Check whether Rc->Ru mapping worked for all the points
    if all(Ru.is_valid() for Ru in Ru_lst_hat):
        print(f'Inverse mapping ok with alpha={alpha:.04f}')
        break
    else:
        print(f'Inverse mapping failed with alpha={alpha:.04f}')
        alpha *= alpha_mult

d:\work\salvador\repo\model_tuner\model_tuner\opt\map_funcs\map_func_1d.py:163: OptimizeWarning: Covariance of the parameters could not be estimated
  par, _ = curve_fit(


Inverse mapping failed with alpha=1.0000
Inverse mapping failed with alpha=0.9000
Inverse mapping failed with alpha=0.8100
Inverse mapping ok with alpha=0.7290


In [12]:
Rc_lst_mixed = NetRegime1DList.mix(Rc_prev_lst, Rc_lst, 0.5)

pop = 'IT2'

for n in range(4):
    Ru = Ru_lst[n][pop].value
    Rc = Rc_lst[n][pop].value
    Rc_prev = Rc_prev_lst[n][pop].value
    Rc_mixed = Rc_lst_mixed[n][pop].value
    print(f'{Rc_prev:.03f} {Rc_mixed:.03f} {Rc:.03f}')

0.880 0.700 0.521
1.320 1.024 0.728
1.760 1.339 0.917
2.200 1.637 1.075
